# Part C: Mage ML Pipeline — Iris Classifier
**Assignment: From Git to Gold**

This notebook runs the complete Mage ML pipeline in Google Colab:
1. **C1**: Run baseline pipeline (`load_iris` -> `check_data_quality` -> `prepare_data` -> `train_model` -> `save_results` & `evaluate_model`)
2. **C2**: Sweep `max_depth` (1 to 5) to demonstrate the `@test` gate
3. **C3**: Launch Mage web UI via Cloudflare tunnel to view and screenshot the DAG branch
4. **C4**: Verify custom data quality block

## 1. Environment Setup
Clone the repository and install `mage-ai` inside a standalone Python 3.11 virtual environment using `uv` (avoids Python 3.13 compilation issues).

In [ ]:
# Clone or pull latest code
import os
if not os.path.exists('/content/data-pipeline'):
    !git clone https://github.com/Ashtsssssh/data-pipeline.git /content/data-pipeline
else:
    %cd /content/data-pipeline
    !git pull

%cd /content/data-pipeline

# Install uv and set up Python 3.11 with mage-ai
!pip install -q uv
!uv venv .venv --python 3.11
!uv pip install --python .venv/bin/python mage-ai scikit-learn pandas matplotlib pyyaml

## 2. Part C1: Run the Pipeline
Run `mage run iris_project iris_classifier --test`.

In [ ]:
%cd /content/data-pipeline
!.venv/bin/mage run iris_project iris_classifier --test

## 3. Part C2: Tests as a Gate (`max_depth = 1, 2, 3, 4, 5`)
Before each run, delete `iris_predictions.csv` and pass `--runtime-vars`.
- `max_depth = 1`: Accuracy is below 90% -> `@test` gate fails -> `save_results` is BLOCKED.
- `max_depth = 2, 3, 4, 5`: Accuracy is above 90% -> `@test` passes -> `iris_predictions.csv` is CREATED.

In [ ]:
import os

results = []

for depth in [1, 2, 3, 4, 5]:
    print(f"\n{'='*25} RUNNING FOR max_depth = {depth} {'='*25}")
    if os.path.exists('iris_predictions.csv'):
        os.remove('iris_predictions.csv')
        
    # Run pipeline
    ret = os.system(f'.venv/bin/mage run iris_project iris_classifier --test --runtime-vars \'{"max_depth": {depth}}\'')
    
    created = os.path.exists('iris_predictions.csv')
    passed = "Passed (>= 0.90)" if created else "Failed (< 0.90)"
    results.append({"max_depth": depth, "Gate": passed, "CSV Created": created})

print("\n" + "="*50)
print("SUMMARY TABLE FOR PART C2 REPORT:")
print("="*50)
import pandas as pd
print(pd.DataFrame(results).to_string(index=False))

## 4. Part C3: Launch Mage UI & Cloudflare Tunnel
This starts the Mage web server on port 6789 and connects Cloudflare tunnel to display your live UI link.

**Instructions:**
1. Run the cell below.
2. Click the `https://...trycloudflare.com` link printed in the output.
3. Go to **Pipelines** -> **`iris_classifier`**.
4. Take a screenshot showing `train_model` branching into `save_results` and `evaluate_model`.

In [ ]:
import subprocess, time, re

# 1. Install cloudflared if needed
!if ! command -v cloudflared &> /dev/null; then \
    curl -s -L https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && \
    chmod +x /usr/local/bin/cloudflared; \
fi

# 2. Stop old instances
!pkill -f mage || true
!pkill -f cloudflared || true

# 3. Start Mage UI server
!cd /content/data-pipeline && nohup .venv/bin/mage start iris_project > /tmp/mage.log 2>&1 &
print("Starting Mage server (waiting 10 seconds)...")
time.sleep(10)

# 4. Launch Cloudflare Tunnel and stream URL
proc = subprocess.Popen("cloudflared tunnel --url http://localhost:6789", shell=True, stderr=subprocess.PIPE, text=True)

for line in proc.stderr:
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        print("\n" + "="*60)
        print("CLICK THIS LINK TO OPEN MAGE UI:")
        print(match.group(0))
        print("="*60 + "\n")
        break